In [1]:
import os
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as transforms
import numpy as np
import cv2
from tqdm import tqdm
import random
from io import BytesIO

src_root = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test"
dst_root = "/kaggle/working/test_augmented_faces"

classes = ["real", "fake"]
resize_size = (224, 224)
normalize = transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)

def to_uint8(tensor):
    img_np = (tensor.numpy().transpose(1, 2, 0) * 0.5 + 0.5) * 255
    return np.clip(img_np, 0, 255).astype(np.uint8)

def save_augmented(image_pil, base_name, aug_type, transform_func, save_dir):
    img = image_pil.resize(resize_size)
    img = transform_func(img)
    img_tensor = TF.to_tensor(img)
    img_tensor = normalize(img_tensor)
    img_np = to_uint8(img_tensor)

    save_path = os.path.join(save_dir, f"{base_name}_{aug_type}.jpg")
    cv2.imwrite(save_path, cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

def lowlight(img): return TF.adjust_brightness(img, 0.3)
def blur(img): return TF.gaussian_blur(img, kernel_size=3)
def jitter(img): return transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)(img)
def flip(img): return TF.hflip(img)
def rotate(img): return TF.rotate(img, angle=15)

def jpeg_compress(img, quality=25):
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    return Image.open(buffer)

def add_shadow(img):
    img_np = np.array(img)
    h, w, _ = img_np.shape
    x1, x2 = np.random.randint(0, w//2), np.random.randint(w//2, w)
    y1, y2 = np.random.randint(0, h//2), np.random.randint(h//2, h)
    mask = np.ones_like(img_np, dtype=np.float32)
    mask[y1:y2, x1:x2] *= 0.3
    img_np = (img_np * mask).astype(np.uint8)
    return Image.fromarray(img_np)

def add_gaussian_noise(img, std=15):
    img_np = np.array(img)
    noise = np.random.normal(0, std, img_np.shape).astype(np.int16)
    noisy = np.clip(img_np + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def salt_pepper(img, amount=0.02):
    img_np = np.array(img)
    num_salt = np.ceil(amount * img_np.size * 0.5).astype(int)
    coords = [np.random.randint(0, i - 1, num_salt) for i in img_np.shape[:2]]
    img_np[coords[0], coords[1]] = 255
    num_pepper = np.ceil(amount * img_np.size * 0.5).astype(int)
    coords = [np.random.randint(0, i - 1, num_pepper) for i in img_np.shape[:2]]
    img_np[coords[0], coords[1]] = 0
    return Image.fromarray(img_np)

def process_class(class_name, limit=5000):
    src_folder = os.path.join(src_root, class_name)
    dst_folder = os.path.join(dst_root, class_name)
    os.makedirs(dst_folder, exist_ok=True)

    img_list = sorted(os.listdir(src_folder))[:limit]
    for i, fname in enumerate(tqdm(img_list, desc=f"Processing {class_name}")):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                path = os.path.join(src_folder, fname)
                image = Image.open(path).convert("RGB")
                base_name = f"{i:05d}"

                save_augmented(image, base_name, "original", lambda x: x, dst_folder)
                save_augmented(image, base_name, "lowlight", lowlight, dst_folder)
                save_augmented(image, base_name, "jpeg", jpeg_compress, dst_folder)
                save_augmented(image, base_name, "gaussnoise", add_gaussian_noise, dst_folder)

            except Exception as e:
                print(f"Error {fname}: {e}")

for cls in classes:
    process_class(cls, limit=5000)

import shutil
shutil.make_archive("/kaggle/working/test_augmented_faces", 'zip', dst_root)

Processing fake: 100%|██████████| 5000/5000 [02:58<00:00, 27.96it/s]


'/kaggle/working/test_augmented_faces.zip'